# IEP-1001 CASE 3 — RAGAS 재측정 v2 (Solar judge, 전 지표)

> **목적**: IEP-2001/2001.1과 공정한 비교를 위해 **전 지표를 Solar judge로 통일**
> **변경점 (v1 대비)**:
> - Context Recall + Precision도 Solar judge로 측정 (기존: llama3.1:8b)
> - retriever k=10 (IEP-2001.1 CANDIDATE_K와 동일 조건)
> - 결과 저장 경로: `results_solar_v2`

## v1 측정 결과 (2026-04-26, 참고용)

| 지표 | llama (v1 기준) | Solar v1 | 비고 |
| :--- | :---: | :---: | :--- |
| Context Recall | 0.8537 | — | v1 미측정 |
| Context Precision | 0.6117 | — | v1 미측정 |
| Faithfulness | 0.4361 | **0.2748** | NaN 38개 |
| Answer Relevancy | 0.6143 | **0.3409** | strictness=1 |

## 실행 순서

```
STEP 1   패키지 설치 → 런타임 자동 재시작
STEP 2   API 키 설정
STEP 3   Config 정의
STEP 4   Google Drive 마운트
STEP 5   ChromaDB 패치 + 임베딩 모델 + ChromaDB 로드
STEP 6   골든 데이터셋 로드
STEP 7   Solar로 Answer 생성 (k=10)  ← k 변경
STEP 8   (선택) 세션 복원
STEP 9   RAGAS 평가 모델 설정
STEP 10  Context Recall + Precision 측정 (Solar judge)  ← 신규
STEP 11  Faithfulness 측정
STEP 12  Answer Relevancy 측정
STEP 13  결과 병합 + 유형별 분석
STEP 14  Solar v2 / Solar v1 / llama 비교
STEP 15  생존 편향 보정 + 최종 저장 + README 마크다운
```


## STEP 1 — 패키지 설치

> ⚠️ 실행 후 런타임이 자동 재시작됩니다. 재시작 후 **STEP 2부터** 실행하세요.
>
> - `numpy==1.26.4` 강제 재설치: Colab 기본 numpy 바이너리 충돌 방지
> - `chromadb==0.5.11` 버전 고정: ChromaDB 패치 코드 검증 버전
> - 나머지 패키지: 버전 고정 없음 → pip 자동 해결

In [ ]:
import subprocess, sys

# numpy 바이너리 충돌 방지 (IEP-4001 방법 B에서 검증된 방식)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', 'numpy', '-y'], capture_output=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'numpy==1.26.4', '--force-reinstall', '-q'], capture_output=True)

!pip install \
    langchain-community \
    langchain-huggingface \
    langchain-openai \
    chromadb==0.5.11 \
    sentence-transformers \
    ragas \
    -q

!pip show openai langchain-openai | grep -E "^(Name|Version)"

print("\u2705 설치 완료 — 런타임 재시작 후 STEP 2부터 실행하세요")

import os
os.kill(os.getpid(), 9)

## STEP 2 — API 키 설정

In [ ]:
import os

# UPSTAGE_API_KEY 환경변수에서 로드
# 설정 방법: Colab Secrets(🔑)에 UPSTAGE_API_KEY 등록 후 아래 주석 해제
# from google.colab import userdata
# os.environ["UPSTAGE_API_KEY"] = userdata.get("UPSTAGE_API_KEY")

UPSTAGE_API_KEY = os.environ.get("UPSTAGE_API_KEY", "")
assert UPSTAGE_API_KEY, "UPSTAGE_API_KEY 환경변수가 설정되지 않았습니다."


## STEP 3 — Config 정의

In [ ]:
# ── 경로 ───────────────────────────────────────────────────────────
BASE_DIR    = "/content/drive/MyDrive/IPCC_data"
CHROMA_DIR  = f"{BASE_DIR}/IEP_1001_CASE3/chroma_db"
GOLDEN_CSV  = f"{BASE_DIR}/IEP_1002/golden_dataset_100.csv"
RESULTS_DIR = f"{BASE_DIR}/IEP_1001_CASE3/results_solar_v2"

# ── 모델 ───────────────────────────────────────────────────────────
COLLECTION_NAME = "ipcc_1001_case3_v1"
EMBED_MODEL     = "jhgan/ko-sroberta-multitask"
SOLAR_BASE_URL  = "https://api.upstage.ai/v1"
SOLAR_MODEL     = "solar-pro3"
TOP_K           = 3    # 답변 생성용 (미사용, CANDIDATE_K로 대체)
CANDIDATE_K     = 10   # IEP-2001.1과 동일 조건
EXPECTED_CHUNKS = 506

# ── 저장 경로 ──────────────────────────────────────────────────────
SAVE_RETRIEVED    = f"{RESULTS_DIR}/iep1001_v2_retrieved.csv"
SAVE_RETRIEVAL    = f"{RESULTS_DIR}/iep1001_v2_retrieval.csv"
SAVE_FAITHFULNESS = f"{RESULTS_DIR}/iep1001_v2_faithfulness.csv"
SAVE_RELEVANCY    = f"{RESULTS_DIR}/iep1001_v2_relevancy.csv"
SAVE_RAW          = f"{RESULTS_DIR}/iep1001_v2_raw.csv"
SAVE_SUMMARY      = f"{RESULTS_DIR}/iep1001_v2_summary.csv"

# ── 비교용 기존 수치 ───────────────────────────────────────────────
LLAMA_RESULTS = {
    "전체":     {"context_recall": 0.8537, "context_precision": 0.6117, "faithfulness": 0.4361, "answer_relevancy": 0.6143},
    "사실 확인": {"context_recall": 0.7381, "context_precision": 0.9133, "faithfulness": 0.6562, "answer_relevancy": 0.5767},
    "비교":     {"context_recall": 0.8600, "context_precision": 0.6733, "faithfulness": 0.3523, "answer_relevancy": 0.5846},
    "의견/예측": {"context_recall": 0.9757, "context_precision": 0.7800, "faithfulness": 0.3908, "answer_relevancy": 0.6822},
    "범위 밖":  {"context_recall": 0.8264, "context_precision": 0.0800, "faithfulness": 0.3261, "answer_relevancy": 0.6066},
}
SOLAR_V1_RESULTS = {
    "전체":     {"faithfulness": 0.2748, "answer_relevancy": 0.3409},
    "사실 확인": {"faithfulness": 0.5125, "answer_relevancy": 0.5910},
    "비교":     {"faithfulness": 0.2818, "answer_relevancy": 0.2670},
    "의견/예측": {"faithfulness": 0.2619, "answer_relevancy": 0.4342},
    "범위 밖":  {"faithfulness": 0.0774, "answer_relevancy": 0.0714},
}

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
print("✅ Config 완료")
print(f"   Judge LLM  : {SOLAR_MODEL} (전 지표)")
print(f"   CANDIDATE_K: {CANDIDATE_K} (IEP-2001.1과 동일 조건)")
print(f"   결과 저장  : {RESULTS_DIR}")


## STEP 4 — Google Drive 마운트

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs(RESULTS_DIR, exist_ok=True)

assert os.path.exists(CHROMA_DIR), f"\u274c ChromaDB 없음: {CHROMA_DIR}"
assert os.path.exists(GOLDEN_CSV), f"\u274c 골든 데이터셋 없음: {GOLDEN_CSV}"

print("\u2705 Drive 마운트 완료")
print(f"   ChromaDB   : {CHROMA_DIR}")
print(f"   골든 데이터셋: {GOLDEN_CSV}")
print(f"   결과 저장   : {RESULTS_DIR}")

## STEP 5 — ChromaDB 패치 + 임베딩 모델 + ChromaDB 로드

> ChromaDB 구버전으로 생성된 DB는 `config_json_str={}`으로 저장되어 `0.5.11`에서 `KeyError: '_type'` 오류 발생.  
> sqlite3로 직접 패치. 이미 패치된 경우 자동 건너뜀.

In [ ]:
import sqlite3
from chromadb.api.configuration import CollectionConfigurationInternal

sqlite_path  = f"{CHROMA_DIR}/chroma.sqlite3"
FIXED_CONFIG = '{"_type": "CollectionConfigurationInternal", "hnsw_configuration": {"_type": "HNSWConfigurationInternal", "space": "cosine", "ef_construction": 100, "ef_search": 10, "num_threads": 4, "resize_factor": 1.2, "batch_size": 100, "sync_threshold": 1000, "M": 16}}'

CollectionConfigurationInternal.from_json_str(FIXED_CONFIG)  # 포맷 사전 검증

conn    = sqlite3.connect(sqlite_path)
cursor  = conn.cursor()
cursor.execute("SELECT name, config_json_str FROM collections;")
rows    = cursor.fetchall()
patched = 0
for name, config in rows:
    if config == "{}":
        cursor.execute("UPDATE collections SET config_json_str = ? WHERE name = ?", (FIXED_CONFIG, name))
        patched += 1
        print(f"   패치 적용: {name}")
    else:
        print(f"   건너뜀 (이미 패치됨): {name}")
conn.commit()
conn.close()
print(f"\u2705 ChromaDB 패치 {'완료' if patched else '불필요 — 이미 정상'}")

In [ ]:
import warnings, torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"   device: {device}")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}
)
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR
)
count = vectorstore._collection.count()
assert count == EXPECTED_CHUNKS, f"\u26a0\ufe0f  청크 수 불일치: 실제 {count}개 (예상 {EXPECTED_CHUNKS}개)"
print(f"\u2705 ChromaDB 로드 완료: {COLLECTION_NAME} ({count}개 청크)")

## STEP 6 — 골든 데이터셋 로드

In [ ]:
import pandas as pd

df_gold = pd.read_csv(GOLDEN_CSV)
assert len(df_gold) == 100,             f"\u26a0\ufe0f  골든 데이터셋 {len(df_gold)}개 (100개여야 함)"
assert 'user_input' in df_gold.columns, "\u274c 'user_input' 컬럼 없음"
assert 'reference'  in df_gold.columns, "\u274c 'reference' 컬럼 없음"

print(f"\u2705 골든 데이터셋 로드 완료: {len(df_gold)}개")
print(df_gold['Type'].value_counts().to_string())

## STEP 7 — Solar로 Answer 생성

> - retriever `k=10` (`CANDIDATE_K`) — IEP-2001.1과 동일 조건
> - 10개마다 체크포인트 자동 저장
> - 예상 소요: 약 3~5분


In [ ]:
from openai import OpenAI
from tqdm.notebook import tqdm
import time

solar_client = OpenAI(
    api_key=UPSTAGE_API_KEY,
    base_url=SOLAR_BASE_URL
)

# IEP-4001 rag.py 와 동일한 프롬프트
RAG_PROMPT = """당신은 IPCC AR6 기후변화 보고서 전문 안내 AI입니다.
반드시 아래 제공된 문서 내용만을 근거로 답변하세요.
문서에 없는 내용은 추측하지 말고 \"해당 내용은 IPCC 보고서에서 찾을 수 없습니다.\"라고 답하세요.

[참고 문서]
{context}

[질문]
{question}

[답변]"""

retriever = vectorstore.as_retriever(search_kwargs={"k": CANDIDATE_K})  # IEP-2001.1과 동일 조건


def generate_answer_solar(question: str, retry: int = 2) -> str:
    docs    = retriever.invoke(question)
    context = "\n\n".join(
        f"[청크 {i+1} | 페이지 {doc.metadata.get('page', 'N/A')}]\n{doc.page_content}"
        for i, doc in enumerate(docs)
    )
    prompt = RAG_PROMPT.format(context=context, question=question)
    for attempt in range(retry + 1):
        try:
            resp = solar_client.chat.completions.create(
                model=SOLAR_MODEL,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=512
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if attempt == retry:
                return f"ERROR: {e}"
            time.sleep(3)


def save_checkpoint(records: list, path: str):
    rows = []
    for r in records:
        row = {
            "id": r["id"], "type": r["type"],
            "user_input": r["user_input"],
            "answer":     r["answer"],
            "reference":  r["reference"],
        }
        for i, ctx in enumerate(r["retrieved_contexts"]):
            row[f"ctx_{i}"] = ctx
        rows.append(row)
    pd.DataFrame(rows).to_csv(path, index=False, encoding='utf-8-sig')


# Retrieval
records = []
for _, row in df_gold.iterrows():
    docs = retriever.invoke(row['user_input'])
    records.append({
        "id":                 row['ID'],
        "type":               row['Type'],
        "user_input":         row['user_input'],
        "retrieved_contexts": [doc.page_content for doc in docs],
        "answer":             "",
        "reference":          row['reference'],
    })

print(f"\u2705 Retrieval 완료: {len(records)}개")
print(f"   컨텍스트 수 확인: {[len(r['retrieved_contexts']) for r in records[:3]]}")

In [ ]:
# Solar Answer 생성 (10개마다 체크포인트 저장)
start_idx = sum(1 for r in records if r['answer'] != "")
if start_idx > 0:
    print(f"\u23e9 {start_idx}개 완료 — {start_idx}번부터 이어서 실행")

start_time = time.time()

for i in tqdm(range(start_idx, len(records)), desc="Solar Answer 생성", initial=start_idx, total=len(records)):
    records[i]['answer'] = generate_answer_solar(records[i]['user_input'])
    if (i + 1) % 10 == 0:
        save_checkpoint(records, SAVE_RETRIEVED)
        elapsed = time.time() - start_time

save_checkpoint(records, SAVE_RETRIEVED)

elapsed     = time.time() - start_time
error_count = sum(1 for r in records if str(r['answer']).startswith('ERROR'))
print(f"   저장: {SAVE_RETRIEVED}")
if error_count:
    print(f"   \u26a0\ufe0f  ERROR: {error_count}개 — RAGAS 평가 시 NaN으로 처리됩니다.")

## STEP 8 — (선택) 세션 재시작 시 중간 저장 파일 복원

> STEP 7이 이미 완료되었다면 이 셀은 **건너뛰세요**.

In [ ]:
import pandas as pd

df_retrieved = pd.read_csv(SAVE_RETRIEVED)
records = []
for _, row in df_retrieved.iterrows():
    contexts = [
        row[f"ctx_{i}"]
        for i in range(TOP_K)
        if pd.notna(row.get(f"ctx_{i}", "")) and row.get(f"ctx_{i}", "") != ""
    ]
    records.append({
        "id":                 row["id"],
        "type":               row["type"],
        "user_input":         row["user_input"],
        "retrieved_contexts": contexts,
        "answer":             row["answer"],
        "reference":          row["reference"],
    })

print(f"\u2705 중간 저장 파일 복원 완료: {len(records)}개")
error_count = sum(1 for r in records if str(r['answer']).startswith('ERROR'))
if error_count:
    print(f"   \u26a0\ufe0f  ERROR {error_count}개 포함 — STEP 7 재실행 권장")

## STEP 9 — RAGAS 평가 모델 설정 (Solar judge)

> - Judge LLM: `solar-pro3` (전 지표)
> - Answer Relevancy 임베딩: `jhgan/ko-sroberta-multitask` (Solar embedding `$.input` 오류로 대체)


In [ ]:
import pandas as pd
from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig
from ragas import EvaluationDataset

# Solar judge LLM
ragas_llm = LangchainLLMWrapper(
    ChatOpenAI(
        model=SOLAR_MODEL,
        openai_api_key=UPSTAGE_API_KEY,
        openai_api_base=SOLAR_BASE_URL,
        temperature=0,
        max_tokens=512
    )
)

# Answer Relevancy 임베딩 — jhgan (STEP 5에서 로드한 객체 재사용)
# Solar embedding은 RAGAS $.input 형식 미지원으로 대체
ragas_embeddings_hf = LangchainEmbeddingsWrapper(embeddings)

run_config = RunConfig(max_workers=2, timeout=120, max_retries=3)

eval_dataset = EvaluationDataset.from_pandas(pd.DataFrame([{
    "user_input":         r['user_input'],
    "retrieved_contexts": r['retrieved_contexts'],
    "response":           r['answer'],
    "reference":          r['reference'],
} for r in records]))

print(f"\u2705 RAGAS 평가 모델 설정 완료")
print(f"   Judge LLM        : {SOLAR_MODEL}")
print(f"   AR 임베딩         : {EMBED_MODEL} (jhgan)")
print(f"   EvaluationDataset: {len(eval_dataset)}개")

## STEP 10 — Context Recall + Precision 측정 (Solar judge)

> 예상 소요: 약 10~20분
> IEP-1001 v1에서 llama judge로 측정했던 Recall/Precision을 Solar로 재측정


In [ ]:
from ragas import evaluate
from ragas.metrics import ContextRecall, ContextPrecision
import time

print(f"▶ Context Recall + Precision 측정 (judge: {SOLAR_MODEL})")
start = time.time()

result_retrieval = evaluate(
    dataset=eval_dataset,
    metrics=[ContextRecall(), ContextPrecision()],
    llm=ragas_llm,
    run_config=run_config,
    batch_size=1,
    raise_exceptions=False,
)
df_retrieval = result_retrieval.to_pandas()
df_retrieval['id']   = [r['id']   for r in records]
df_retrieval['type'] = [r['type'] for r in records]
df_retrieval.to_csv(SAVE_RETRIEVAL, index=False, encoding='utf-8-sig')

elapsed = time.time() - start
rc_mean = df_retrieval['context_recall'].mean(skipna=True)
cp_mean = df_retrieval['context_precision'].mean(skipna=True)
rc_nan  = df_retrieval['context_recall'].isna().sum()
cp_nan  = df_retrieval['context_precision'].isna().sum()
print(f"   Context Recall    : {rc_mean:.4f}  (NaN: {rc_nan}개)")
print(f"   Context Precision : {cp_mean:.4f}  (NaN: {cp_nan}개)")
print(f"   llama 기준        : Recall {LLAMA_RESULTS['전체']['context_recall']:.4f} / "
      f"Precision {LLAMA_RESULTS['전체']['context_precision']:.4f}")
print(f"   저장: {SAVE_RETRIEVAL}")


## STEP 11 — Faithfulness 측정

> 예상 소요: 약 10~20분 


In [ ]:
from ragas import evaluate
from ragas.metrics import Faithfulness
import time

print(f"\u25b6 Faithfulness 평가 시작 (judge: {SOLAR_MODEL})")
start = time.time()

result_faith = evaluate(
    dataset=eval_dataset,
    metrics=[Faithfulness()],
    llm=ragas_llm,
    embeddings=ragas_embeddings_hf,
    run_config=run_config,
    batch_size=1,
    raise_exceptions=False,
)
df_faith = result_faith.to_pandas()
df_faith['id']   = [r['id']   for r in records]
df_faith['type'] = [r['type'] for r in records]
df_faith.to_csv(SAVE_FAITHFULNESS, index=False, encoding='utf-8-sig')

elapsed    = time.time() - start
faith_mean = df_faith['faithfulness'].mean()
faith_nan  = df_faith['faithfulness'].isna().sum()
print(f"   Faithfulness : {faith_mean:.4f}  (NaN: {faith_nan}개)")
print(f"   llama 기준   : {LLAMA_RESULTS['전체']['faithfulness']:.4f}  "
      f"({'\u2191' if faith_mean > LLAMA_RESULTS['전체']['faithfulness'] else '\u2193'} "
      f"{faith_mean - LLAMA_RESULTS['전체']['faithfulness']:+.4f})")
print(f"   저장: {SAVE_FAITHFULNESS}")

## STEP 12 — Answer Relevancy 측정

> 예상 소요: 약 3~5분 
>
> ⚠️ Solar API 호환 이슈:
> - `strictness=1`: Solar `n=1` 제약 (기본값 3 사용 불가)
> - `ragas_embeddings_hf`: Solar embedding `$.input` 형식 오류로 jhgan 사용


In [ ]:
from ragas.metrics import AnswerRelevancy

# strictness=1: Solar n=1 제약 우회 (기본값 3은 n=3 요청으로 Solar에서 400 오류)
metric_ar = AnswerRelevancy(strictness=1)

print(f"\u25b6 Answer Relevancy 평가 시작 (strictness=1, jhgan embedding)")
start = time.time()

result_rel = evaluate(
    dataset=eval_dataset,
    metrics=[metric_ar],
    llm=ragas_llm,
    embeddings=ragas_embeddings_hf,
    run_config=run_config,
    batch_size=1,
    raise_exceptions=False,
)
df_rel = result_rel.to_pandas()
df_rel['id']   = [r['id']   for r in records]
df_rel['type'] = [r['type'] for r in records]
df_rel.to_csv(SAVE_RELEVANCY, index=False, encoding='utf-8-sig')

elapsed  = time.time() - start
rel_mean = df_rel['answer_relevancy'].mean()
rel_nan  = df_rel['answer_relevancy'].isna().sum()
print(f"   Answer Relevancy : {rel_mean:.4f}  (NaN: {rel_nan}개)")
print(f"   llama 기준       : {LLAMA_RESULTS['전체']['answer_relevancy']:.4f}")

## STEP 13 — 결과 병합 + 유형별 분석


In [ ]:
import numpy as np

ALL_METRICS = ['context_recall', 'context_precision', 'faithfulness', 'answer_relevancy']

df_raw = pd.DataFrame({
    'id':                [r['id']        for r in records],
    'type':              [r['type']       for r in records],
    'user_input':        [r['user_input'] for r in records],
    'context_recall':    df_retrieval['context_recall'].values,
    'context_precision': df_retrieval['context_precision'].values,
    'faithfulness':      df_faith['faithfulness'].values,
    'answer_relevancy':  df_rel['answer_relevancy'].values,
})

overall_mean = df_raw[ALL_METRICS].mean(skipna=True).round(4)
overall_nan  = df_raw[ALL_METRICS].isna().sum()
summary      = df_raw.groupby('type')[ALL_METRICS].mean().round(4)

print("전체 평균 (NaN 제외):")
for m in ALL_METRICS:
    print(f"  {m:<22}: {overall_mean[m]:.4f}  (NaN: {overall_nan[m]}개)")

print("\n유형별 평균:")
print(summary.to_string())


## STEP 14 — Solar v2 / Solar v1 / llama 비교


In [ ]:
types_order = ['사실 확인', '비교', '의견/예측', '범위 밖']

print("=" * 74)
print("  전 지표 비교: Solar v2 (이번) vs Solar v1 vs llama3.1:8b")
print("=" * 74)

for metric in ALL_METRICS:
    print(f"\n[{metric}]")
    print(f"  {'유형':<10} {'Solar v2':>10} {'Solar v1':>10}(diff) {'llama':>10}(diff)")
    print("  " + "-" * 56)
    for t in types_order:
        v2  = summary.loc[t, metric] if t in summary.index else float('nan')
        v1  = SOLAR_V1_RESULTS.get(t, {}).get(metric, float('nan'))
        llm = LLAMA_RESULTS.get(t, {}).get(metric, float('nan'))
        d_v1  = f"{v2-v1:+.4f}"  if not (np.isnan(v2) or np.isnan(v1))  else "  N/A"
        d_llm = f"{v2-llm:+.4f}" if not (np.isnan(v2) or np.isnan(llm)) else "  N/A"
        print(f"  {t:<10} {v2:>10.4f} {v1:>10.4f}({d_v1}) {llm:>10.4f}({d_llm})")
    v2_all  = overall_mean[metric]
    v1_all  = SOLAR_V1_RESULTS.get('전체', {}).get(metric, float('nan'))
    llm_all = LLAMA_RESULTS['전체'][metric]
    d_v1_a  = f"{v2_all-v1_all:+.4f}"  if not np.isnan(v1_all) else "  N/A"
    d_llm_a = f"{v2_all-llm_all:+.4f}"
    print("  " + "-" * 56)
    print(f"  {'전체':<10} {v2_all:>10.4f} {v1_all:>10.4f}({d_v1_a}) {llm_all:>10.4f}({d_llm_a})")


## STEP 15 — 생존 편향 보정 + 최종 저장 + README 마크다운


In [ ]:
# 생존 편향 보정
print("=" * 60)
print("  생존 편향 보정 (NaN → 0점, 전체 100개 기준)")
print("=" * 60)
print(f"  {'지표':<22} {'낙관적(NaN제외)':>15} {'유효샘플':>8} {'보수적(전체100개)':>18}")
print("  " + "-" * 65)
for m in METRICS:
    opt = overall_mean[m]
    vn  = 100 - overall_nan[m]
    pes = round(opt * vn / 100, 4)
    print(f"  {m:<22} {opt:>15.4f} {vn:>8}개 {pes:>18.4f}")

In [ ]:
# 최종 저장 (전 지표 Solar v2 실측값)
df_raw.to_csv(SAVE_RAW, index=False, encoding='utf-8-sig')

df_s = summary.copy()
df_s.loc['전체'] = overall_mean
df_s.to_csv(SAVE_SUMMARY, encoding='utf-8-sig')

print("✅ 저장 완료")
print(f"  raw     → {SAVE_RAW}")
print(f"  summary → {SAVE_SUMMARY}")


In [ ]:
# README / IEP 문서용 마크다운 출력
ALL_MEANS = {
    'context_recall':    overall_mean['context_recall'],    # Solar v2 실측
    'context_precision': overall_mean['context_precision'], # Solar v2 실측
    'faithfulness':      overall_mean['faithfulness'],
    'answer_relevancy':  overall_mean['answer_relevancy'],
}

types_order = ['사실 확인', '비교', '의견/예측', '범위 밖']

md  = "| 유형 | Context Recall | Context Precision | Faithfulness | Answer Relevancy |\n"
md += "| :--- | :---: | :---: | :---: | :---: |\n"
for t in types_order:
    cr = summary.loc[t, 'context_recall']    if t in summary.index else float('nan')
    cp = summary.loc[t, 'context_precision'] if t in summary.index else float('nan')
    fa = summary.loc[t, 'faithfulness']      if t in summary.index else float('nan')
    ar = summary.loc[t, 'answer_relevancy']  if t in summary.index else float('nan')
    md += f"| {t} | {cr:.4f} | {cp:.4f} | {fa:.4f} | {ar:.4f} |\n"
md += (f"| **전체** | **{ALL_MEANS['context_recall']:.4f}** "
       f"| **{ALL_MEANS['context_precision']:.4f}** "
       f"| **{ALL_MEANS['faithfulness']:.4f}** "
       f"| **{ALL_MEANS['answer_relevancy']:.4f}** |\n")

print("[README 마크다운 — IEP-1001 CASE 3 (Solar v2, 전 지표)]")
print(md)

# 생존 편향 보정 마크다운
nan_md  = "| 지표 | 낙관적 (NaN 제외) | 유효 샘플 | 보수적 (전체 100개) |\n"
nan_md += "| :--- | :---: | :---: | :---: |\n"
for m in ALL_METRICS:
    opt = overall_mean[m]
    vn  = 100 - overall_nan[m]
    pes = round(opt * vn / 100, 4)
    nan_md += f"| {m} | {opt:.4f} | {vn}개 | **{pes:.4f}** |\n"
print("[생존 편향 보정 마크다운]")
print(nan_md)

print("=" * 60)
print("IEP-1001 CASE3 RAGAS v2 재측정 완료 (Solar judge, 전 지표)")
print("=" * 60)
for m, v in ALL_MEANS.items():
    llama_v = LLAMA_RESULTS['전체'][m]
    v1_v    = SOLAR_V1_RESULTS.get('전체', {}).get(m, float('nan'))
    v1_str  = f"Solar v1: {v1_v:.4f}" if not np.isnan(v1_v) else f"llama: {llama_v:.4f}"
    print(f"  {m:<22}: {v:.4f}  ({v1_str})")

print()
print("⚠️  주의사항")
print("   - Answer Relevancy: strictness=1 (Solar n=1 제약), jhgan 임베딩")
print("   - Context Recall/Precision: Solar judge (v1 llama 수치와 직접 비교 불가)")
